# Communication Dropout Diagnostics

Read one or more JSON outputs from `scripts/diagnose_joint_happo.py` and compare the core joint performance metrics:

- scout recall: mean and std
- confirmation recall: mean and std
- full-confirm success: rate and binary std
- final confidence: mean and std
- final coverage: mean and std

Add the diagnostic JSON files you want to compare to `JSON_FILES` in the first code cell.

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

CWD = Path.cwd().resolve()
if (CWD / 'scripts' / 'diagnose_joint_happo.py').is_file():
    PROJECT_ROOT = CWD
elif (CWD.parent / 'scripts' / 'diagnose_joint_happo.py').is_file():
    PROJECT_ROOT = CWD.parent
else:
    PROJECT_ROOT = Path('..').resolve()
NOTEBOOK_DIR = PROJECT_ROOT / 'notebooks'

# Add diagnose_joint_happo.py JSON outputs here.
JSON_FILES = [
    "../outputs/uav4_ugv3_1km_256_190_900_hid128__coms00_1000_1009.json",
    "../outputs/uav4_ugv3_1km_256_190_900_hid128__coms03_1000_1009.json",
    "../outputs/uav4_ugv3_1km_256_190_900_hid128__coms05_1000_1009.json",
    "../outputs/uav4_ugv3_1km_256_190_900_hid128__coms07_1000_1009.json",
    "../outputs/uav4_ugv3_1km_256_190_900_hid128__coms09_1000_1009.json",
    # PROJECT_ROOT / 'outputs/900k/joint_suvivor_900k_300_warmup_1000_1099.json',
]

# Optional convenience fallback while exploring interactively.
if not JSON_FILES:
    JSON_FILES = sorted((PROJECT_ROOT / 'outputs').glob('**/joint*.json'))

def _resolve_json_path(path):
    path = Path(path).expanduser()
    candidates = [path]
    if not path.is_absolute():
        candidates.extend([PROJECT_ROOT / path, NOTEBOOK_DIR / path])
    for candidate in candidates:
        if candidate.is_file():
            return candidate.resolve()
    return path


JSON_FILES = [_resolve_json_path(path) for path in JSON_FILES]
missing = [path for path in JSON_FILES if not path.is_file()]

if missing:
    raise FileNotFoundError('Missing diagnostic JSON files:\n' + '\n'.join(str(p) for p in missing))
if not JSON_FILES:
    raise FileNotFoundError('No joint diagnostic JSON files configured or found under outputs/**/*.json')

print(f'Loaded file list: {len(JSON_FILES)} JSON file(s)')
for path in JSON_FILES:
    print(' -', path.relative_to(PROJECT_ROOT) if path.is_relative_to(PROJECT_ROOT) else path)


In [ ]:
def _summary_value(summary, *keys, default=float('nan')):
    for key in keys:
        if key in summary:
            return summary[key]
    return default


def _success_rate(summary):
    rate = _summary_value(summary, 'full_confirm_success_rate', 'success_rate')
    if pd.isna(rate):
        percent = _summary_value(summary, 'full_confirm_success_percent')
        rate = percent / 100.0 if not pd.isna(percent) else float('nan')
    return rate


def _success_std(summary, rate):
    if pd.isna(rate):
        return float('nan')
    episodes = _summary_value(summary, 'episodes')
    if pd.isna(episodes) or episodes <= 0:
        return float('nan')
    return float(np.sqrt(max(rate * (1.0 - rate), 0.0)))


def _label_from_payload(path, payload):
    scenario = payload.get('scenario', {})
    dropout = scenario.get('comms_dropout', None)
    mode = scenario.get('comms_dropout_mode', None)
    if dropout is None:
        return path.stem
    return f'{path.stem} | dropout={dropout:g}, mode={mode or "?"}'


records = []
for path in JSON_FILES:
    payload = json.loads(path.read_text())
    summary = payload.get('summary', {})
    scenario = payload.get('scenario', {})
    success_rate = _success_rate(summary)
    records.append({
        'file': str(path),
        'label': _label_from_payload(path, payload),
        'episodes': _summary_value(summary, 'episodes'),
        'comms_dropout': scenario.get('comms_dropout', float('nan')),
        'comms_dropout_mode': scenario.get('comms_dropout_mode', 'unknown'),
        'scout_recall_mean': _summary_value(summary, 'mean_scout_recall'),
        'scout_recall_std': _summary_value(summary, 'std_scout_recall'),
        'confirm_recall_mean': _summary_value(summary, 'mean_confirm_recall'),
        'confirm_recall_std': _summary_value(summary, 'std_confirm_recall'),
        'success_mean': success_rate,
        'success_std': _success_std(summary, success_rate),
        'success_count': _summary_value(summary, 'full_confirm_success_count'),
        'final_confidence_mean': _summary_value(summary, 'mean_final_confidence'),
        'final_confidence_std': _summary_value(summary, 'std_final_confidence'),
        'final_coverage_mean': _summary_value(summary, 'mean_final_coverage_fraction'),
        'final_coverage_std': _summary_value(summary, 'std_final_coverage_fraction'),
    })

metrics = pd.DataFrame.from_records(records)
metrics


## Compact Table

The table below formats each metric as `mean +/- std` for quick comparison.

In [ ]:
def _pm(mean, std):
    if pd.isna(mean):
        return 'n/a'
    if pd.isna(std):
        return f'{mean:.3f}'
    return f'{mean:.3f} +/- {std:.3f}'


compact = pd.DataFrame({
    'label': metrics['label'],
    'episodes': metrics['episodes'].astype('Int64'),
    'scout recall': [
        _pm(mean, std)
        for mean, std in zip(metrics['scout_recall_mean'], metrics['scout_recall_std'])
    ],
    'confirm recall': [
        _pm(mean, std)
        for mean, std in zip(metrics['confirm_recall_mean'], metrics['confirm_recall_std'])
    ],
    'success': [
        _pm(mean, std)
        for mean, std in zip(metrics['success_mean'], metrics['success_std'])
    ],
    'success count': metrics['success_count'].astype('Int64'),
    'final confidence': [
        _pm(mean, std)
        for mean, std in zip(metrics['final_confidence_mean'], metrics['final_confidence_std'])
    ],
    'final coverage': [
        _pm(mean, std)
        for mean, std in zip(metrics['final_coverage_mean'], metrics['final_coverage_std'])
    ],
}).set_index('label')

compact


## Plots

If the inputs form a communication-dropout sweep, the metrics are drawn as errorbar curves over dropout probability. Otherwise the notebook falls back to grouped bar-style comparisons.

In [ ]:
PLOT_METRICS = [
    ('scout_recall', 'Scout Recall'),
    ('confirm_recall', 'Confirmation Recall'),
    ('success', 'Full-Confirm Success'),
    ('final_confidence', 'Final Confidence'),
    ('final_coverage', 'Final Coverage'),
]

plot_df = metrics.copy()
plot_df['comms_dropout'] = pd.to_numeric(plot_df['comms_dropout'], errors='coerce')
has_dropout_x = plot_df['comms_dropout'].notna().all() and plot_df['comms_dropout'].nunique() > 1

fig, axes = plt.subplots(2, 3, figsize=(14, 7), constrained_layout=True)
axes = axes.ravel()

if has_dropout_x:
    plot_df = plot_df.sort_values(['comms_dropout', 'label'])
    x = plot_df['comms_dropout'].to_numpy(dtype=float)
    for ax, (prefix, title) in zip(axes, PLOT_METRICS):
        mean = plot_df[f'{prefix}_mean'].to_numpy(dtype=float)
        std = plot_df[f'{prefix}_std'].to_numpy(dtype=float)
        ax.errorbar(x, mean, yerr=std, marker='o', capsize=4, linewidth=2)
        ax.set_xlabel('communication dropout probability')
        ax.set_ylabel('fraction')
        ax.set_title(title)
        ax.set_ylim(0.0, 1.02)
        ax.grid(alpha=0.25)
else:
    plot_df = plot_df.reset_index(drop=True)
    x = np.arange(len(plot_df))
    labels = plot_df['label'].tolist()
    for ax, (prefix, title) in zip(axes, PLOT_METRICS):
        mean = plot_df[f'{prefix}_mean'].to_numpy(dtype=float)
        std = plot_df[f'{prefix}_std'].to_numpy(dtype=float)
        ax.bar(x, mean, yerr=std, capsize=4, color='#4f7df3', alpha=0.78)
        ax.set_xticks(x)
        ax.set_xticklabels(labels, rotation=35, ha='right')
        ax.set_ylabel('fraction')
        ax.set_title(title)
        ax.set_ylim(0.0, 1.02)
        ax.grid(axis='y', alpha=0.25)

for ax in axes[len(PLOT_METRICS):]:
    ax.set_axis_off()

fig.suptitle('Joint Diagnostics Under Communication Dropout', fontsize=14)
plt.show()
